In [1]:
from pathlib import Path
import json

import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

daily_config = json.loads(
    (REPORT_DIR / "daily_listening_config.json").read_text(
        encoding="utf-8"
    )
)

assert daily_config["user_day_duplicates_checked"]
assert daily_config["duplicate_user_days"] == 0

logs_dir = PROJECT_ROOT / daily_config["output_directory"]
parquet_pattern = str(logs_dir / "**" / "*.parquet")

temp_dir = PROJECT_ROOT / "data" / "duckdb_tmp"
temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(config={
    "threads": 2,
    "memory_limit": "2GB",
    "temp_directory": str(temp_dir),
    "max_temp_directory_size": "2GB",
})

window_metadata = pd.DataFrame({
    "cohort": ["historical_train", "later_evaluation"],
    "cutoff": pd.to_datetime([
        daily_config["training_window"][1],
        daily_config["evaluation_window"][1],
    ]),
})

con.register("window_metadata", window_metadata)

window_metadata

,cohort,cutoff
0,historical_train,2016-11-30
1,later_evaluation,2017-02-28


## Aggregate Daily Logs into User-Level Features

In [2]:
listening_feature_sql = """
WITH daily AS (
    SELECT
        l.*,
        w.cutoff,
        (
            l.num_25 + l.num_50 + l.num_75
            + l.num_985 + l.num_100
        ) AS song_plays
    FROM read_parquet(?, hive_partitioning = true) AS l
    INNER JOIN window_metadata AS w
        ON l.cohort = w.cohort
    WHERE l.cohort = ?
)
SELECT
    cohort,
    msno,
    COUNT(*) AS recorded_days_30d,
    SUM(song_plays) AS recorded_song_plays_30d,
    SUM(num_100) AS completed_song_plays_30d,
    SUM(num_unq) AS sum_daily_unique_songs_30d,
    SUM(total_secs_clean) AS valid_listening_seconds_30d,
    COUNT(total_secs_clean) AS valid_duration_days_30d,
    SUM(duration_anomaly) AS flagged_duration_days_30d,
    AVG(total_secs_clean) AS avg_seconds_per_valid_day_30d,
    SUM(num_100)::DOUBLE
        / NULLIF(SUM(song_plays), 0) AS completion_ratio_30d,
    DATE_DIFF(
        'day', MAX(log_day), MAX(cutoff)
    ) AS days_since_last_log_30d,
    COUNT(*) FILTER (
        WHERE log_day >= cutoff - INTERVAL '6 days'
    ) AS recorded_days_recent_7d,
    COUNT(*) FILTER (
        WHERE log_day BETWEEN cutoff - INTERVAL '13 days'
                          AND cutoff - INTERVAL '7 days'
    ) AS recorded_days_previous_7d,
    COALESCE(
        SUM(song_plays) FILTER (
            WHERE log_day >= cutoff - INTERVAL '6 days'
        ), 0
    ) AS recorded_plays_recent_7d,
    COALESCE(
        SUM(song_plays) FILTER (
            WHERE log_day BETWEEN cutoff - INTERVAL '13 days'
                              AND cutoff - INTERVAL '7 days'
        ), 0
    ) AS recorded_plays_previous_7d
FROM daily
GROUP BY cohort, msno
"""

In [3]:
feature_results = []

for cohort in ["historical_train", "later_evaluation"]:
    print(f"Building features for {cohort}...", flush=True)

    result = con.execute(
        listening_feature_sql,
        [parquet_pattern, cohort],
    ).df()

    feature_results.append(result)

listening_features = pd.concat(
    feature_results,
    ignore_index=True,
)

listening_features["recorded_plays_7d_change"] = (
    listening_features["recorded_plays_recent_7d"]
    - listening_features["recorded_plays_previous_7d"]
)

print("Listening feature table shape:", listening_features.shape)

Building features for historical_train...
Building features for later_evaluation...
Listening feature table shape: (1426418, 17)


In [4]:
assert not listening_features.duplicated(["cohort", "msno"]).any()

assert listening_features["recorded_days_30d"].between(1, 30).all()

assert listening_features["days_since_last_log_30d"].between(
    0, 29
).all()

assert listening_features["recorded_days_recent_7d"].between(
    0, 7
).all()

assert listening_features["recorded_days_previous_7d"].between(
    0, 7
).all()

assert (
    listening_features["valid_duration_days_30d"]
    + listening_features["flagged_duration_days_30d"]
    == listening_features["recorded_days_30d"]
).all()

completion_ratios = listening_features[
    "completion_ratio_30d"
].dropna()

assert completion_ratios.between(0, 1).all()

listening_feature_summary = (
    listening_features.groupby("cohort")
    .agg(
        users_with_logs=("msno", "size"),
        mean_recorded_days=("recorded_days_30d", "mean"),
        mean_recorded_plays=("recorded_song_plays_30d", "mean"),
        flagged_duration_days=("flagged_duration_days_30d", "sum"),
    )
    .reset_index()
)

listening_feature_summary

,cohort,users_with_logs,mean_recorded_days,mean_recorded_plays,flagged_duration_days
0,historical_train,667477,16.968787,679.108721,2384.0
1,later_evaluation,758941,16.711162,661.984659,3052.0


## Join Listening Features to the Complete Modeling Cohorts

Retain every labeled user. Fill recorded-count features with zero
when no logs are available, while keeping averages, ratios, duration
totals, and recency missing. Preserve a listening-log coverage indicator.

In [5]:
zero_fill_columns = [
    "recorded_days_30d",
    "recorded_song_plays_30d",
    "completed_song_plays_30d",
    "sum_daily_unique_songs_30d",
    "valid_duration_days_30d",
    "flagged_duration_days_30d",
    "recorded_days_recent_7d",
    "recorded_days_previous_7d",
    "recorded_plays_recent_7d",
    "recorded_plays_previous_7d",
    "recorded_plays_7d_change",
]

def combine_features(transaction_path, cohort):
    transaction_data = con.read_parquet(
        str(transaction_path)
    ).df()

    behavior_data = (
        listening_features.loc[
            listening_features["cohort"] == cohort
        ]
        .drop(columns="cohort")
        .copy()
    )

    assert (
        set(transaction_data.columns)
        & set(behavior_data.columns)
    ) == {"msno"}

    combined = transaction_data.merge(
        behavior_data,
        on="msno",
        how="left",
        validate="one_to_one",
        indicator="listening_join",
    )

    combined["has_listening_logs_30d"] = (
        combined["listening_join"] == "both"
    ).astype("int8")

    combined = combined.drop(columns="listening_join")

    combined[zero_fill_columns] = (
        combined[zero_fill_columns]
        .fillna(0)
        .astype("int64")
    )

    assert len(combined) == len(transaction_data)
    assert combined["msno"].is_unique
    assert combined["is_churn"].equals(transaction_data["is_churn"])

    return combined

historical_combined = combine_features(
    PROCESSED_DIR / "historical_training_features_cutoff_20161130.parquet",
    "historical_train",
)

later_combined = combine_features(
    PROCESSED_DIR / "candidate_transaction_features_v2_cutoff_20170228.parquet",
    "later_evaluation",
)

print("Historical combined shape:", historical_combined.shape)
print("Later combined shape:", later_combined.shape)

Historical combined shape: (882127, 25)
Later combined shape: (970960, 25)


In [6]:
combined_summary_rows = []

for cohort, dataset in [
    ("historical_train", historical_combined),
    ("later_evaluation", later_combined),
]:
    assert dataset["recorded_days_30d"].between(0, 30).all()

    assert (
        dataset["valid_duration_days_30d"]
        + dataset["flagged_duration_days_30d"]
        == dataset["recorded_days_30d"]
    ).all()

    no_logs = dataset["has_listening_logs_30d"].eq(0)

    assert dataset.loc[
        no_logs, zero_fill_columns
    ].eq(0).all().all()

    unknown_columns = [
        "valid_listening_seconds_30d",
        "avg_seconds_per_valid_day_30d",
        "completion_ratio_30d",
        "days_since_last_log_30d",
    ]

    assert dataset.loc[
        no_logs, unknown_columns
    ].isna().all().all()

    combined_summary_rows.append({
        "cohort": cohort,
        "users": len(dataset),
        "users_with_logs": int(
            dataset["has_listening_logs_30d"].sum()
        ),
        "users_without_logs": int(no_logs.sum()),
        "feature_count": len(
            dataset.columns.difference(["msno", "is_churn"])
        ),
        "churn_rate": dataset["is_churn"].mean(),
    })

combined_feature_summary = pd.DataFrame(combined_summary_rows)
combined_feature_summary

,cohort,users,users_with_logs,users_without_logs,feature_count,churn_rate
0,historical_train,882127,667477,214650,23,0.096627
1,later_evaluation,970960,758941,212019,23,0.089942


In [7]:
sql_dir = PROJECT_ROOT / "sql"
sql_dir.mkdir(parents=True, exist_ok=True)

con.from_df(historical_combined).write_parquet(
    str(PROCESSED_DIR / "historical_combined_features_20161130.parquet"),
    overwrite=True,
)

con.from_df(later_combined).write_parquet(
    str(PROCESSED_DIR / "later_combined_features_20170228.parquet"),
    overwrite=True,
)

listening_feature_summary.to_csv(
    REPORT_DIR / "listening_feature_summary.csv",
    index=False,
)

combined_feature_summary.to_csv(
    REPORT_DIR / "combined_feature_summary.csv",
    index=False,
)

(sql_dir / "listening_features_30d.sql").write_text(
    listening_feature_sql.strip() + "\n",
    encoding="utf-8",
)

combined_feature_columns = [
    column
    for column in historical_combined.columns
    if column not in ["msno", "is_churn"]
]

assert set(combined_feature_columns) == (
    set(later_combined.columns) - {"msno", "is_churn"}
)

feature_config = {
    "transaction_window_days": 90,
    "listening_window_days": 30,
    "training_cutoff": "2016-11-30",
    "evaluation_cutoff": "2017-02-28",
    "features": combined_feature_columns,
    "zero_fill_columns": zero_fill_columns,
    "coverage_indicator": "has_listening_logs_30d",
    "duration_upper_bound": 86400,
    "threshold_sensitivity_checked": False,
    "daily_unique_song_counts_are_summed": True,
    "monthly_distinct_song_count_available": False,
}

(REPORT_DIR / "combined_feature_config.json").write_text(
    json.dumps(feature_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Combined features, summaries, SQL, and configuration saved.")

Combined features, summaries, SQL, and configuration saved.


## Listening Behavior Features

Aggregated 24,009,061 extracted daily activity records into user-level listening features and joined them to the transaction feature tables.

The combined datasets retain all 882,127 historical training users and 970,960 later evaluation users, with 23 candidate features per user: seven transaction features, fifteen listening features, and one listening-log coverage indicator.

Listening features describe recorded days, playback volume, completed-play ratio, valid listening duration, recency, and changes between the most recent seven days and the preceding seven days.

- Historical training: 667,477 users have window logs; 214,650 do not.
- Later evaluation: 758,941 users have window logs; 212,019 do not.

Recorded-count features are filled with zero for users without logs, with a coverage indicator retained. Duration totals, averages, ratios, and recency remain missing when unavailable. Summed daily unique-song counts are not interpreted as monthly distinct songs.

The transaction window spans 90 days and the listening window spans 30 days, ending at each cohort's feature cutoff. The provisional duration threshold still requires sensitivity checks. Predictive improvements from listening features have not yet been evaluated.

User-level feature tables remain local and are excluded from Git.

See [the feature notebook](notebooks/11_listening_features.ipynb), [aggregation SQL](sql/listening_features_30d.sql), [cohort summary](reports/combined_feature_summary.csv), and [feature configuration](reports/combined_feature_config.json).